# AutoResearch Experiment Platform — Colab training

Runtime → Change runtime type → **GPU**.

This notebook only runs the `training/` pipeline (no FastAPI / dashboard).

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__)

In [ ]:
# Clone (or pull) the platform repo
!git clone https://github.com/justSamarthings/autoresearch-platform.git
%cd autoresearch-platform

In [ ]:
%cd training
!pip install -q -r requirements.txt
# Optional FA3 kernels (ignored if install/runtime fails; train.py falls back to SDPA)
!pip install -q "kernels>=0.11.7" || true

In [ ]:
# Full TinyStories download (~673MB) + tokenizer training
!python prepare.py

In [ ]:
# Optional short smoke run (~30s training budget). Comment out for full 5-minute experiment.
import os
os.environ["AUTORESEARCH_TIME_BUDGET"] = "30"
os.environ["AUTORESEARCH_NO_COMPILE"] = "1"  # faster first-run smoke
!python train.py

In [ ]:
# Full 5-minute default experiment (TIME_BUDGET=300)
# import os
# for k in ("AUTORESEARCH_TIME_BUDGET", "AUTORESEARCH_NO_COMPILE"):
#     os.environ.pop(k, None)
# !python train.py

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path("artifacts/checkpoints")
res_dir = Path("artifacts/results")
ckpts = sorted(ckpt_dir.glob("*.pt"))
results = sorted(res_dir.glob("*.json"))
assert ckpts, "No checkpoint found"
assert results, "No result JSON found"
print("latest checkpoint:", ckpts[-1])
print("latest result:", results[-1])
print(json.loads(results[-1].read_text())["val_bpb"])

!python ../scripts/verify_checkpoint.py {ckpts[-1]}